See https://github.com/EO-DataHub/eodhp-guide/blob/main/examples/uc9-data-discovery-and-analysis/CommercialDataOrdering.ipynb
and Gemma, using the bottom bit of this notebook to get different quotes for different licenses + clipped/unclipped

NEEDS REWORKING

# Setup - Load an API Key and Libraries

## Storing and Loading Your API Key Securely

To securely store and use your API key in this notebook, follow these steps:

1. **Generate an API key**:
   - Follow instructions in the Getting Started documentation for Hub APIs to generate a Hub API Key.

2. **Create a `.env` File**:
   - In the same directory as this notebook, create a file named `.env`.
   - Add the following line to the file, replacing `<your_api_key>` with your actual API key:
     ```plaintext
     API_KEY=<your_api_key>
     ```
   - Save the file.

3. **Load the Key in the Notebook**:
   - The following code snippet is already included in this notebook to load the key securely:
     ```python
     import os
     from dotenv import load_dotenv
     
     load_dotenv(".env")  # Ensure the path matches your `.env` file location
     key = os.getenv("API_KEY")
     ```
   - This will load the `API_KEY` from your `.env` file into the `key` variable.

4. **Keep the `.env` File Secure**:
   - Do not share your `.env` file or API key.
   - If you do accidentally share your key, delete it in the workspaces UI and request a new one.

By following these steps, you can securely store and use your API key without exposing it in the notebook.

## Setting Up the Workspace and Environment

To ensure the notebook works correctly, you need to configure the `workspace` and `environment` variables. Follow these steps:
1. **Ensure your workspace is configured for commercial orders**
   - Follow instructions in the Hub documentation under `Data - Commercial` to link your commercial account to your workspace.
2. **Set the `workspace` Variable**:
   - In the following cell, the `workspace` variable should be set to the name of the workspace you wish to order data for.
   - You can find the workspace name in the site UI.
3. **Set the `environment` Variable**
   - In the following cell, The environment variable determines which environment you are working in. It can only be one of the following values:
     - prod (Production)
     - staging (Staging)
     - test (Testing)
     - dev (Development)

In [1]:
import os
from dotenv import load_dotenv

workspace = "gemmaprodtest"
environment = "prod"

load_dotenv("gem-auth.txt")
api_key = os.getenv("API_KEY")

if environment == "prod":
    platform_domain = "https://eodatahub.org.uk"
elif environment == "staging":
    platform_domain = "https://staging.eodatahub.org.uk"
elif environment == "test":
    platform_domain = "https://test.eodatahub.org.uk"
elif environment == "dev":
    platform_domain = "https://dev.eodatahub.org.uk"
else:
    raise ValueError("Invalid environment. Choose from 'prod', 'staging', 'test', or 'dev'.")

In [2]:
!pip install pystac-client xarray rasterio

# Create a pystac-client Client for the EODH Catalogue

The EODH catalogue can be viewed through a UI on the Hub under the `Catalogue` tab.

Our STAC catalogue can also be browsed programatically using tools such as pystac-client, using your api_key for authorisation. In the following example, our client is scoped to the Planet catalogue to refine a search for some Planet commercial data.

In [3]:
from pystac_client import Client

# Limit scope of the top-level catalogue to Planet
rc_url = f"{platform_domain}/api/catalogue/stac/catalogs/commercial/catalogs/airbus"

# Create STAC client
stac_client = Client.open(rc_url, headers={"Authorization": f"Bearer {api_key}"})

In [ ]:
geom = {
    "type": "Polygon",
    "coordinates": [
        [
            [9.6, 57.1],
            [9.6, 57.0],
            [9.8, 56.9],
            [9.8, 57.0],
            [9.6, 57.1],
        ]
    ],
}
search = stac_client.search(
    max_items=10,
    collections=['PSScene'],
    intersects=geom,
)
for item in search.items():
    print(item.get_self_href())

In [ ]:
import requests

item_href = "https://staging.eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/planet/collections/PSScene/items/20250217_101155_07_24c7"
url = f"{item_href}/quote"
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "coordinates": [
        [
            [9.6, 57.1],
            [9.6, 57.0],
            [9.8, 56.9],
            [9.8, 57.0],
            [9.6, 57.1]
        ]
    ]
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

In [ ]:
import requests

item_href = "https://staging.eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/planet/collections/PSScene/items/20250217_101155_07_24c7"
url = f"{item_href}/order"
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "productBundle": "General use",
    "coordinates": [
        [
            [9.6, 57.1],
            [9.6, 57.0],
            [9.8, 56.9],
            [9.8, 57.0],
            [9.6, 57.1]
        ]
    ]
}

response = requests.post(url, headers=headers, json=data)

print("Status Code:", response.status_code)
print("Response:", response.json())

location_header = response.headers.get('Location')
print("Item will shortly be available at:", location_header)

In [ ]:
from pystac import Item
from pystac_client import CollectionClient

data_i_ordered_earlier = f"{platform_domain}/api/catalogue/stac/catalogs/user/catalogs/{workspace}/catalogs/commercial-data/catalogs/planet"
stac_client = Client.open(data_i_ordered_earlier, headers={"Authorization": f"Bearer {api_key}"})

In [ ]:
ordered_item = next(stac_client.get_items("20250217_101155_07_24c7"))
ordered_item

In [ ]:
asset = ordered_item.get_assets()["20250217_101155_07_24c7_3B_AnalyticMS_clip.tif"]
asset

In [ ]:
import urllib3
from io import BytesIO

resp = urllib3.request("GET", asset.href, headers={"Authorization": f"Bearer {api_key}"})
resp.data[0:100]

# Obtain a Quote for Airbus Optical Data

Search and ordering capabilities are also in place for Airbus Optical items. Once an item of interest is found, you may obtain a quote for the item via a POST request to /quote following the item href.

Change the `item_href` to any valid link to a STAC item in our Airbus commercial catalogue to browse prices. Change the `coordinates` to an AOI to clip the item if required.

Note: Coordinates are to be provided in longitude/latitude WGS84, and the polygon must be closed. Coordinates follow OGC GeoJSON specification.

Change the `licence` to another supported licence if required. Supported licences are:
  - `Standard`
  - `Background Layer`
  - `Standard + Background Layer`
  - `Academic`
  - `Media Licence`
  - `Standard Multi End-Users (2-5)`
  - `Standard Multi End-Users (6-10)`
  - `Standard Multi End-Users (11-30)`
  - `Standard Multi End-Users (>30)`

In [ ]:
# Tower Hamlets BBOX
[
    [-0.08017620447063, 51.4845636802758], 
    [0.009932571058582, 51.4845636802758], 
    [0.009932571058582, 51.5446504200296], 
    [-0.08017620447063, 51.5446504200296], 
    [-0.08017620447063, 51.4845636802758]
]
# Camden BBOX
[
    [-0.213437487930831, 51.5126101149581], 
    [-0.105323965744685, 51.5126101149581], 
    [-0.105323965744685, 51.572981384291], 
    [-0.213437487930831, 51.572981384291], 
    [-0.213437487930831, 51.5126101149581]
]
# Brent BBOX
[
    [-0.335562160691732, 51.5276559502351], 
    [-0.19145849063202, 51.5276559502351], 
    [-0.19145849063202, 51.6003731988738], 
    [-0.335562160691732, 51.6003731988738], 
    [-0.335562160691732, 51.5276559502351]
]

In [ ]:
collection_id = "airbus_phr_data"

In [169]:
img_id = "ACQ_PNEO4_03813510086486"

In [170]:
import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/quote" #<--- specifies quote
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "licence": "Standard Multi End-Users (2-5)",
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

Status Code 200
Response  {'value': 1987.2, 'units': 'EUR'}


In [171]:
import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/quote" #<--- specifies quote
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "licence": "Standard Multi End-Users (6-10)",
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

Status Code 200
Response  {'value': 2384.64, 'units': 'EUR'}


In [172]:
import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/quote" #<--- specifies quote
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "licence": "Standard Multi End-Users (2-5)",
    "coordinates": [
        [
            [-0.213437487930831, 51.5126101149581], 
            [-0.105323965744685, 51.5126101149581], 
            [-0.105323965744685, 51.572981384291], 
            [-0.213437487930831, 51.572981384291], 
            [-0.213437487930831, 51.5126101149581]
        ]
    ]
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

Status Code 200
Response  {'value': 576.0, 'units': 'EUR'}


In [173]:
import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/quote" #<--- specifies quote
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "licence": "Standard Multi End-Users (6-10)",
    "coordinates": [
        [
            [-0.213437487930831, 51.5126101149581], 
            [-0.105323965744685, 51.5126101149581], 
            [-0.105323965744685, 51.572981384291], 
            [-0.213437487930831, 51.572981384291], 
            [-0.213437487930831, 51.5126101149581]
        ]
    ]
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

Status Code 200
Response  {'value': 691.2, 'units': 'EUR'}
